# 03 — Curadoria e persistência Silver

Objetivo: tipar e padronizar os três arquivos sem apagar observações. Entradas: CSVs do volume e perfil do notebook 02. Saídas: pedidos, itens e clientes em tabelas Delta Silver. Pré-requisitos: confirmar chaves e esquema no perfil. Evidências: resultados de conversão, contagens antes/depois e telas das três tabelas no catálogo.

A política padrão bloqueia a carga em caso de chaves nulas ou duplicadas, pois escolher uma linha arbitrária alteraria o grão. Valores inválidos de campos não chave tornam-se nulos por conversão tolerante e são contados; os CSVs originais continuam disponíveis. Outliers e fretes negativos não são removidos.


In [ ]:
from pyspark.sql import functions as F
import re

dbutils.widgets.text("catalogo", "")
catalogo = dbutils.widgets.get("catalogo").strip() or spark.sql("SELECT current_catalog()").first()[0]
assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", catalogo), "Informe um catálogo válido no widget."
assert catalogo != "hive_metastore", "Use o catálogo do workspace com Unity Catalog."
spark.conf.set("spark.sql.session.timeZone", "America/Sao_Paulo")
origem = f"/Volumes/{catalogo}/olist_mvp_raw/originais"
silver = f"{catalogo}.olist_mvp_silver"
gold = f"{catalogo}.olist_mvp_gold"
arquivos = {
    "pedidos": "olist_orders_dataset.csv",
    "itens": "olist_order_items_dataset.csv",
    "clientes": "olist_customers_dataset.csv",
}
colunas_esperadas = {
    "pedidos": ["order_id", "customer_id", "order_status", "order_purchase_timestamp",
                "order_approved_at", "order_delivered_carrier_date",
                "order_delivered_customer_date", "order_estimated_delivery_date"],
    "itens": ["order_id", "order_item_id", "product_id", "seller_id",
              "shipping_limit_date", "price", "freight_value"],
    "clientes": ["customer_id", "customer_unique_id", "customer_zip_code_prefix",
                 "customer_city", "customer_state"],
}
print(f"Catálogo: {catalogo}; volume: {origem}")


In [ ]:
def ler_origem(nome):
    caminho = f"{origem}/{arquivos[nome]}"
    quadro = (spark.read.option("header", "true")
              .option("inferSchema", "false")
              .option("sep", ",")
              .option("encoding", "UTF-8")
              .option("mode", "FAILFAST")
              .csv(caminho))
    assert quadro.columns == colunas_esperadas[nome], (
        f"Esquema inesperado em {arquivos[nome]}: {quadro.columns}"
    )
    return quadro


## Leitura e verificações que impedem multiplicação de pedidos


In [ ]:
brutos = {nome: ler_origem(nome) for nome in arquivos}
for nome, chave in [("pedidos", ["order_id"]), ("itens", ["order_id", "order_item_id"]),
                    ("clientes", ["customer_id"])]:
    quadro = brutos[nome]
    for coluna in chave:
        assert quadro.filter(F.col(coluna).isNull() | (F.trim(F.col(coluna)) == "")).count() == 0, (
            f"Chave ausente em {nome}.{coluna}; investigar antes de gravar Silver"
        )
    assert quadro.groupBy(*chave).count().filter("count > 1").count() == 0, (
        f"Chave duplicada em {nome}; investigar antes de gravar Silver"
    )


## Conversões explícitas


In [ ]:
p = brutos["pedidos"]
pedidos_silver = p.select(
    F.trim("order_id").alias("order_id"),
    F.trim("customer_id").alias("customer_id"),
    F.lower(F.trim("order_status")).alias("order_status"),
    *[F.expr(f"try_cast({c} AS TIMESTAMP)").alias(c) for c in [
        "order_purchase_timestamp", "order_approved_at",
        "order_delivered_carrier_date", "order_delivered_customer_date"]],
    F.expr("try_cast(order_estimated_delivery_date AS DATE)").alias("order_estimated_delivery_date")
)
i = brutos["itens"]
itens_silver = i.select(
    F.trim("order_id").alias("order_id"),
    F.expr("try_cast(order_item_id AS INT)").alias("order_item_id"),
    F.trim("product_id").alias("product_id"),
    F.trim("seller_id").alias("seller_id"),
    F.expr("try_cast(shipping_limit_date AS TIMESTAMP)").alias("shipping_limit_date"),
    F.expr("try_cast(price AS DECIMAL(18,2))").alias("price"),
    F.expr("try_cast(freight_value AS DECIMAL(18,2))").alias("freight_value")
)
c = brutos["clientes"]
clientes_silver = c.select(
    F.trim("customer_id").alias("customer_id"),
    F.trim("customer_unique_id").alias("customer_unique_id"),
    F.when(F.trim("customer_zip_code_prefix") != "",
           F.lpad(F.trim("customer_zip_code_prefix"), 5, "0")).alias("customer_zip_code_prefix"),
    F.lower(F.trim("customer_city")).alias("customer_city"),
    F.upper(F.trim("customer_state")).alias("customer_state")
)


## Diagnóstico das conversões

A comparação entre texto presente na origem e nulo após tipagem mostra erros de conversão. Nulos de data real em pedidos ainda não entregues são reportados separadamente. Se houver falha de conversão material, registre a contagem e avalie seu impacto antes da análise.


In [ ]:
def falhas_de_tipo(quadro, coluna, tipo):
    presente = F.col(coluna).isNotNull() & (F.trim(F.col(coluna)) != "")
    return quadro.filter(presente & F.expr(f"try_cast({coluna} AS {tipo})").isNull()).count()

tipos = {
    "pedidos": {
        "order_purchase_timestamp": "TIMESTAMP",
        "order_approved_at": "TIMESTAMP",
        "order_delivered_carrier_date": "TIMESTAMP",
        "order_delivered_customer_date": "TIMESTAMP",
        "order_estimated_delivery_date": "DATE",
    },
    "itens": {
        "order_item_id": "INT",
        "shipping_limit_date": "TIMESTAMP",
        "price": "DECIMAL(18,2)",
        "freight_value": "DECIMAL(18,2)",
    },
}
falhas = [(nome, coluna, falhas_de_tipo(brutos[nome], coluna, tipo))
          for nome, campos in tipos.items() for coluna, tipo in campos.items()]
display(spark.createDataFrame(falhas, ["tabela", "coluna", "falhas_conversao"]))


## Reconciliação de linhas e regras de qualidade


In [ ]:
tabelas = {"pedidos": pedidos_silver, "itens": itens_silver, "clientes": clientes_silver}
reconciliacao = []
for nome, quadro in tabelas.items():
    entrada = brutos[nome].count()
    saida = quadro.count()
    assert entrada == saida, f"Perda inesperada em {nome}: {entrada} → {saida}"
    reconciliacao.append((nome, entrada, saida, saida - entrada))
display(spark.createDataFrame(reconciliacao, ["tabela", "linhas_origem", "linhas_silver", "diferenca"]))

assert itens_silver.filter(F.col("order_item_id").isNull()).count() == 0, "Identificador de item inválido"
display(pedidos_silver.groupBy("order_status").agg(
    F.count("*").alias("pedidos"),
    F.sum(F.when(F.col("order_delivered_customer_date").isNull(), 1).otherwise(0)).alias("sem_data_real")
))
display(itens_silver.agg(
    F.sum(F.when(F.col("price") < 0, 1).otherwise(0)).alias("precos_negativos"),
    F.sum(F.when(F.col("freight_value") < 0, 1).otherwise(0)).alias("fretes_negativos"),
    F.sum(F.when(F.col("price").isNull(), 1).otherwise(0)).alias("precos_ausentes"),
    F.sum(F.when(F.col("freight_value").isNull(), 1).otherwise(0)).alias("fretes_ausentes")
))


## Persistência idempotente e catálogo

O modo overwrite substitui o conteúdo da tabela a cada execução com a mesma origem; não duplica registros. Guarde a captura das tabelas no Explorador de Catálogo após esta célula. Os comentários de tabela e coluna complementam o catálogo integral no README.


In [ ]:
comentarios = {
    "pedidos": ("Pedidos tipados, uma linha por order_id.", {
        "order_id": "Identificador do pedido; origem pedidos.",
        "customer_id": "Cliente vinculado ao pedido; origem pedidos.",
        "order_status": "Status normalizado em minúsculas; origem pedidos.",
        "order_purchase_timestamp": "Instante da compra; origem pedidos.",
        "order_approved_at": "Instante da aprovação; origem pedidos.",
        "order_delivered_carrier_date": "Instante da entrega à transportadora; origem pedidos.",
        "order_delivered_customer_date": "Instante da entrega ao cliente; origem pedidos.",
        "order_estimated_delivery_date": "Data prevista da entrega; origem pedidos.",
    }),
    "itens": ("Itens tipados, uma linha por order_id e order_item_id.", {
        "order_id": "Pedido ao qual pertence o item; origem itens.",
        "order_item_id": "Sequência do item no pedido; origem itens.",
        "product_id": "Identificador do produto; origem itens.",
        "seller_id": "Identificador do vendedor; origem itens.",
        "shipping_limit_date": "Limite de envio do vendedor; origem itens.",
        "price": "Preço do item em reais; origem itens.",
        "freight_value": "Frete do item em reais; origem itens.",
    }),
    "clientes": ("Clientes e localidade informada, uma linha por customer_id.", {
        "customer_id": "Identificador do cliente por pedido; origem clientes.",
        "customer_unique_id": "Identificador persistente do cliente; origem clientes.",
        "customer_zip_code_prefix": "Cinco dígitos iniciais do CEP; origem clientes.",
        "customer_city": "Cidade em minúsculas; origem clientes.",
        "customer_state": "UF em maiúsculas; origem clientes.",
    }),
}
for nome, quadro in tabelas.items():
    tabela = f"{silver}.{nome}"
    quadro.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(tabela)
    assert spark.table(tabela).count() == quadro.count(), f"Falha de persistência: {tabela}"
    descricao, colunas = comentarios[nome]
    spark.sql(f"COMMENT ON TABLE {tabela} IS '{descricao}'")
    for coluna, comentario in colunas.items():
        spark.sql(f"ALTER TABLE {tabela} ALTER COLUMN {coluna} COMMENT '{comentario}'")
    print(f"{tabela}: {quadro.count()} linhas persistidas")
